# Clasificación Binaria de Mortalidad en un Hospital

**Objetivo:** La idea de este proyecto es poder predecir la mortalidad de un paciente que ingresa a una hospitalización:

### Base de datos
- `Hospital Management Data`: https://www.kaggle.com/datasets/deeplumiere/hospital-management-data/data

- `Licencia`: MIT - https://www.mit.edu/~amini/LICENSE.md 



### Archivos de Origen
- `visit_encounters.csv`: Información de la hospitalización.
- `clinical_details.csv`: Información detallada de la hospitalización.
- `hospital_billing.csv`: Facturación de la hospitalización.
- `patients.csv`: Información del paciente.
- `pharmacy_prescriptions.csv`: Medicametos recetados durante la hospitalización.



## Librerias y Configuración

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## Análisis Exploratorio de cada Dataset

Para entender el contexto de cada uno de los datasets y asi poder definir el dataset final entonces haremos un análisis individual para ver su cantidad, calidad y distribución de los datos.

In [ ]:
# Cargar de los datasets
patients   = pd.read_csv("../data/patients.csv")
encounters = pd.read_csv("../data/visits_encounters.csv")
clinical   = pd.read_csv("../data/clinical_details.csv")
billing    = pd.read_csv("../data/hospital_billing.csv")
pharmacy   = pd.read_csv("../data/pharmacy_prescriptions.csv")

datasets = {
    "patients": patients,
    "visits_encounters": encounters,
    "clinical_details": clinical,
    "hospital_billing": billing,
    "pharmacy_prescriptions": pharmacy
}

resumen = []
for name, df in datasets.items():
    resumen.append({
        "Dataset": name,
        "Filas" : df.shape[0],
        "Columnas" : df.shape[1],
        "Faltantes" : round(df.isnull().mean().mean()*100,2)
    })
resumen_df = pd.DataFrame(resumen)
resumen_df

,dataset,Filas,Columnas,Faltantes
0,patients,5000,21,1.46
1,visits_encounters,13214,18,4.81
2,clinical_details,13214,22,3.53
3,hospital_billing,13214,18,5.28
4,pharmacy_prescriptions,26715,16,0.32


## Patients

In [ ]:
patients.head()

,patient_id,first_name,last_name,gender,date_of_birth,age,blood_group,marital_status,race_ethnicity,residential_zone,socioeconomic_tier,primary_insurance_provider,insurance_policy_id,contact_phone,emergency_contact_phone,street_address,zip_code,baseline_bmi,smoker_status,baseline_charlson_comorbidity_index,registration_date
0,PAT-10001,Mark,Johnson,Male,1945-11-05,77,O+,Married,Hispanic,Suburban,High,Medicare,INS-887201,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,35172,35.70,Former,1,2021-10-02
1,PAT-10002,Renee,Blair,Female,1982-12-16,40,B-,Married,Caucasian,Suburban,Middle,Self-Pay,NaN,+1-516-318-4959,NaN,3419 Amanda Gardens Apt. 764,84387,30.00,Never,3,2021-08-24
2,PAT-10003,Melissa,Robinson,Female,2004-10-09,18,O+,Single,African American,Suburban,Middle,BlueCross,INS-560337,903-405-6413x95376,423-988-4969x653,12269 Paul Ranch,61849,21.10,Never,1,2021-05-30
3,PAT-10004,Kendra,Maddox,Female,1942-02-10,80,A+,Divorced,Caucasian,Urban,Low,Medicare,INS-123247,9182451462,NaN,15430 Natalie Common Apt. 718,16984,32.60,Former,4,2021-01-31
4,PAT-10005,Betty,Alvarez,Female,2004-03-10,18,A+,Single,Hispanic,Urban,Middle,BlueCross,INS-384806,+1-624-889-6383,+1-557-287-1331x50983,031 Michele Cliffs,67892,33.30,Current,1,2021-02-18


In [11]:
patients.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 21 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   patient_id                           5000 non-null   str    
 1   first_name                           5000 non-null   str    
 2   last_name                            5000 non-null   str    
 3   gender                               5000 non-null   str    
 4   date_of_birth                        5000 non-null   str    
 5   age                                  5000 non-null   int64  
 6   blood_group                          5000 non-null   str    
 7   marital_status                       5000 non-null   str    
 8   race_ethnicity                       5000 non-null   str    
 9   residential_zone                     5000 non-null   str    
 10  socioeconomic_tier                   5000 non-null   str    
 11  primary_insurance_provider           5000

In [12]:
patients.describe(include='all')

,patient_id,first_name,last_name,gender,date_of_birth,age,blood_group,marital_status,race_ethnicity,residential_zone,socioeconomic_tier,primary_insurance_provider,insurance_policy_id,contact_phone,emergency_contact_phone,street_address,zip_code,baseline_bmi,smoker_status,baseline_charlson_comorbidity_index,registration_date
count,5000,5000,5000,5000,5000,5000.00,5000,5000,5000,5000,5000,5000,4598,4714,4709,5000,5000.00,4703.00,4746,5000.00,5000
unique,5000,589,915,2,4563,NaN,8,4,5,3,3,6,4587,4714,4709,5000,NaN,NaN,3,NaN,550
top,PAT-10001,Michael,Smith,Female,1971-12-20,NaN,O+,Married,Caucasian,Urban,Middle,BlueCross,INS-728138,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,NaN,NaN,Never,NaN,2021-09-16
freq,1,117,99,2514,3,NaN,1899,2296,2966,2722,2443,1318,2,1,1,1,NaN,NaN,2994,NaN,20
mean,NaN,NaN,NaN,NaN,NaN,44.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49882.47,28.39,NaN,1.97,NaN
std,NaN,NaN,NaN,NaN,NaN,23.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28891.06,4.38,NaN,1.69,NaN
min,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,507.00,15.00,NaN,0.00,NaN
25%,NaN,NaN,NaN,NaN,NaN,27.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24534.50,25.40,NaN,1.00,NaN
50%,NaN,NaN,NaN,NaN,NaN,47.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49866.00,28.50,NaN,2.00,NaN
75%,NaN,NaN,NaN,NaN,NaN,65.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74702.50,31.40,NaN,3.00,NaN


## Visit Encounters

In [ ]:
encounters.head()

In [ ]:
encounters.info(verbose=True)

In [ ]:
encounters.describe(include='all')

## Clinical Details

In [ ]:
clinical.head()

In [ ]:
clinical.info(verbose=True)

In [ ]:
clinical.describe(include='all')

## Hospital Billing

In [ ]:
billing.head()

In [ ]:
billing.info(verbose=True)

In [ ]:
billing.describe(include='all')

## Pharmacy

In [ ]:
pharmacy.head()

In [ ]:
pharmacy.info(verbose=True)

In [ ]:
pharmacy.describe(include='all')

## Relaciones entre datasets y decisión de unión

El dataset principal que usaremos es donde se encuentra la variable in_hospital_mortality que es la que nos va a servir para predecir de ahí para adelante se usará uniones con las otras tablas con la siguiente llave:

- `visit_encounters.csv`: Principal.
- `clinical_details.csv`: encounter_id.
- `hospital_billing.csv`: encounter_id.
- `patients.csv`: patient_id.
- `pharmacy_prescriptions.csv`: encounter_id.

Adicionalmente en la tabla de pharmacy al tener la posibilidad de tener varias prescripciones medicas para una misma hospitalización lo que se aplicará son funciones agregadas

In [ ]:
pharm_agg = (
    pharmacy
    .groupby("encounter_id")
    .agg(
        n_meds               = ("pharmacy_order_id", "count"),
        n_high_risk_meds     = ("high_risk_medication_flag", "sum"),
        total_drug_cost      = ("total_drug_cost", "sum"),
        patient_copay_drugs  = ("patient_copay_drug_cost", "sum"),
      )
    .reset_index()
)


df = (
    encounters
    .merge(clinical,  on=["encounter_id", "patient_id"], how="left", suffixes=("", "_clin"))
    .merge(billing,  on=["encounter_id", "patient_id"], how="left", suffixes=("", "_bill"))
    .merge(patients,   on="patient_id", how="left", suffixes=("", "_pat"))
    .merge(pharm_agg, on="encounter_id", how="left")
)

print(df.shape)         


(13214, 78)
in_hospital_mortality
0   0.90
1   0.10
Name: proportion, dtype: float64


In [16]:
df.head()

,encounter_id,patient_id,encounter_type,admission_datetime,discharge_datetime,length_of_stay_days,admission_urgency,department,attending_physician_id,icu_admitted,icu_days,discharge_disposition,news2_acuity_score,revisit_count,previous_encounter_id,days_since_last_discharge,readmission_30d,in_hospital_mortality,clinical_record_id,primary_diagnosis_code,...,age,blood_group,marital_status,race_ethnicity,residential_zone,socioeconomic_tier,primary_insurance_provider,insurance_policy_id,contact_phone,emergency_contact_phone,street_address,zip_code,baseline_bmi,smoker_status,baseline_charlson_comorbidity_index,registration_date,n_meds,n_high_risk_meds,total_drug_cost,patient_copay_drugs
0,ENC-100001,PAT-10001,Emergency,2023-03-30 00:00:00,2023-04-03 07:12:00,4.30,Emergency,General Medicine,DOC-106,0,0.00,Skilled Nursing Facility,1,1,NaN,NaN,0,0,CLN-100001,N39.0,...,77,O+,Married,Hispanic,Suburban,High,Medicare,INS-887201,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,35172,35.70,Former,1,2021-10-02,3,0,100.72,18.13
1,ENC-100002,PAT-10001,Outpatient,2023-04-06 12:00:00,2023-04-06 16:19:12,0.18,Elective,General Medicine,DOC-151,0,0.00,Home,2,2,ENC-100001,3.20,0,0,CLN-100002,J44.1,...,77,O+,Married,Hispanic,Suburban,High,Medicare,INS-887201,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,35172,35.70,Former,1,2021-10-02,1,0,56.40,10.15
2,ENC-100003,PAT-10001,Outpatient,2023-04-18 04:19:12,2023-04-18 13:55:12,0.40,Elective,Cardiology,DOC-111,0,0.00,Home,9,3,ENC-100002,11.50,1,0,CLN-100003,I50.9,...,77,O+,Married,Hispanic,Suburban,High,Medicare,INS-887201,533-521-8196x001,790.683.8637x94026,51161 Abbott Rapids,35172,35.70,Former,1,2021-10-02,1,0,14.40,2.59
3,ENC-100004,PAT-10002,Emergency,2023-03-28 00:00:00,2023-04-07 19:12:00,10.80,Emergency,Cardiology,DOC-115,0,0.00,Home,7,1,NaN,NaN,1,0,CLN-100004,I21.9,...,40,B-,Married,Caucasian,Suburban,Middle,Self-Pay,NaN,+1-516-318-4959,NaN,3419 Amanda Gardens Apt. 764,84387,30.00,Never,3,2021-08-24,1,0,4.73,4.73
4,ENC-100005,PAT-10002,Inpatient,2023-04-09 12:00:00,2023-04-12 16:48:00,3.20,Urgent,General Medicine,DOC-134,0,0.00,Home,0,2,ENC-100004,1.70,1,0,CLN-100005,J18.9,...,40,B-,Married,Caucasian,Suburban,Middle,Self-Pay,NaN,+1-516-318-4959,NaN,3419 Amanda Gardens Apt. 764,84387,30.00,Never,3,2021-08-24,3,0,89.55,89.55


In [ ]:
#Exportar el dataframe final a un archivo CSV
df.to_csv("../data/merged_df.csv", index=False)